# Example 4.3 / 1: Filtering an Indexing Set

Companion notebook for **4.3 Indexing Expressions** in the AMPL lab Documentation, section "Filtering an Indexing Set". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Filtering an Indexing Set

The condition after `:` keeps only members for which the logical expression is true. The example
defines a derived set and retrieves it with `to_list()`.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PRODUCTS;
param margin {PRODUCTS};

set PROFITABLE = {product in PRODUCTS: margin[product] > 0};

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PRODUCTS"] = ["bands", "coils", "plate"]
ampl.param["margin"] = {"bands": 15, "coils": -2, "plate": 8}

In [5]:
# run.py
print(ampl.set["PROFITABLE"].to_list())
print(ampl.get_data("{p in PRODUCTS: margin[p] >= 8}").to_list())

['bands', 'plate']
['bands', 'plate']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PRODUCTS := bands coils plate;
param margin := bands 15  coils -2  plate 8;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display PROFITABLE;
display {p in PRODUCTS: margin[p] >= 8};

set PROFITABLE := bands plate;

set {p in PRODUCTS: margin[p] >= 8}  := bands plate;

